# EVCS Statewide Dynamic Demand Forecasting (Stanford ROLAND GNN Framework)
## Canonical Final Demonstration & Inference Pipeline

**Objective**: Execute the canonical dynamic GNN pipeline to forecast EV charging station demand across all 33 districts of Telangana (934 nodes).

### Key Methodological Highlights:
1. **Dynamic Stanford ROLAND Architecture**: Combines `ResidualEdgeConv` (spatial GNN message passing over district-wise subgraphs) with `GRUUpdater` (recurrent state transitions).
2. **Active Node Masking**: Masks out uncommissioned stations based on commissioning dates, mapping inactive node dynamic features to scaled zero representations.
3. **Zero-Masking Spatial Coordinates**: Zeroes spatial coordinates (`snap.x[:, 0] = 0.0`, `snap.x[:, 1] = 0.0`) to enforce pure topological spatial message passing.
4. **Canonical Pre-trained Model**: Defaults to loading `final_model/roland_final_submission_80_02.pt` (or `roland_final_weights_80_02.pt`) for immediate inference.
5. **Strict Chronological Memory Carryover**: `H_val = H_train.detach()` and `H_test = H_val.detach()` to block backpropagation gradient leakage across splits.


In [1]:
import os
import sys
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to sys.path for clean relative imports
sys.path.append(os.path.abspath(".."))

import config
from datasets.dataset import TemporalEVCSDataset
from models.roland_model import ROLANDModel
from training.trainer_temporal import ROLANDTemporalTrainer
from training.trainer import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Target Execution Device: {device}")


PyTorch Version: 2.12.1+cpu
Target Execution Device: cpu


In [2]:
# Feature configuration overrides matching Untitled.ipynb canonical section
config.USE_LAGS = True
config.USE_ROLLING = False
config.USE_GROWTH = False
config.USE_SEASONAL = True
config.USE_YEAR = False
config.USE_CHARGER_AGE = True
config.USE_EV_REGISTRATIONS = False
config.USE_POI = True
config.USE_POPULATION = False
config.USE_GRAPH_FEATURES = True

print("=================== Canonical Pipeline Configuration ===================")
print(f"Train Months:      {config.TRAIN_MONTHS} (Months 1-26)")
print(f"Val Months:        {config.VAL_MONTHS} (Months 27-31)")
print(f"Test Months:       {config.TEST_MONTHS} (Months 32-36)")
print(f"Hidden Dimension:  {config.HIDDEN_DIM} (256)")
print(f"Learning Rate:     {config.LR} (0.0005)")
print(f"Optimizer:         {config.OPTIMIZER_NAME} (Adam)")
print(f"Feature Flags:     LAGS={config.USE_LAGS}, ROLLING={config.USE_ROLLING}, GROWTH={config.USE_GROWTH}, SEASONAL={config.USE_SEASONAL}, POI={config.USE_POI}")
print("=========================================================================")


=================== Canonical Pipeline Configuration ===================
Train Months:      26 (Months 1-26)
Val Months:        5 (Months 27-31)
Test Months:       5 (Months 32-36)
Hidden Dimension:  256 (256)
Learning Rate:     0.0005 (0.0005)
Optimizer:         Adam (Adam)
Feature Flags:     LAGS=True, ROLLING=False, GROWTH=False, SEASONAL=True, POI=True


In [3]:
print("Loading processed dynamic graph dataset...")
dataset = TemporalEVCSDataset(
    data_dir="../processed",
    fit_scale_indices=list(range(config.TRAIN_MONTHS)),
    scale_features=True
)

# Apply zero-masking for Latitude and Longitude (indices 0 and 1) exactly as in canonical pipeline
for snap in dataset.snapshots:
    snap.x[:, 0] = 0.0
    snap.x[:, 1] = 0.0

print(f"Successfully loaded {len(dataset)} monthly graph snapshots.")
print(f"Number of nodes per snapshot: {dataset[0].num_nodes}")
print(f"Node feature matrix shape (data.x): {dataset[0].x.shape}")
print(f"Target Mean (Units, Load): {dataset.target_mean.tolist()}")
print(f"Target Std  (Units, Load): {dataset.target_std.tolist()}")


Loading processed dynamic graph dataset...
Successfully loaded 36 monthly graph snapshots.
Number of nodes per snapshot: 934
Node feature matrix shape (data.x): torch.Size([934, 52])
Target Mean (Units, Load): [12823.947265625, 435.5880126953125]
Target Std  (Units, Load): [19467.287109375, 1962.5701904296875]


In [4]:
in_channels = dataset[0].x.size(1)  # 52
hidden_dim = 256

model = ROLANDModel(
    in_channels=in_channels,
    hidden_channels=hidden_dim * 2,  # 512
    out_channels=hidden_dim          # 256
).to(device)

# Load pre-trained canonical model checkpoint
ckpt_path = os.path.join("../final_model", "roland_final_submission_80_02.pt")
weights_path = os.path.join("../final_model", "roland_final_weights_80_02.pt")

if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    print(f"Successfully loaded canonical pre-trained weights from '{weights_path}' with ZERO shape mismatches!")
elif os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    print(f"Successfully loaded canonical submission checkpoint from '{ckpt_path}' with ZERO shape mismatches!")
else:
    raise FileNotFoundError("Fatal Error: Canonical trained model checkpoints not found in final_model/")


Successfully loaded canonical pre-trained weights from '../final_model\roland_final_weights_80_02.pt' with ZERO shape mismatches!


In [5]:
trainer = ROLANDTemporalTrainer(
    hidden_dim=hidden_dim,
    num_epochs=config.EPOCHS,
    lr=config.LR,
    patience=config.PATIENCE,
    device=str(device)
)
model.eval()

snapshots = [snap.to(device) for snap in dataset.snapshots]
train_snaps = snapshots[:config.TRAIN_MONTHS]
val_snaps = snapshots[config.TRAIN_MONTHS : config.TRAIN_MONTHS + config.VAL_MONTHS]
test_snaps = snapshots[config.TRAIN_MONTHS + config.VAL_MONTHS :]
num_nodes = snapshots[0].num_nodes

with torch.no_grad():
    # 1. Replay Training Sequence
    H_train = torch.zeros(num_nodes, hidden_dim, device=device)
    for snap in train_snaps:
        _, H_train = model(snap, H_train)
        
    # 2. Replay Validation Sequence (detached state)
    H_val = H_train.detach()
    val_loss, val_metrics, _ = trainer.validate_sequence(
        model,
        val_snaps,
        H_train,
        target_mean=dataset.target_mean.to(device),
        target_std=dataset.target_std.to(device)
    )
        
    # 3. Evaluate Test Sequence (detached state)
    test_loss, test_metrics = trainer.evaluate_sequence(
        model,
        test_snaps,
        H_val,
        target_mean=dataset.target_mean.to(device),
        target_std=dataset.target_std.to(device)
    )

print("\n====================================================================")
print("                   CANONICAL EVALUATION RESULTS                     ")
print("====================================================================")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Test Loss:       {test_loss:.4f}")
print(f"Test RMSE:       {test_metrics['avg_rmse']:.4f}")
print(f"Test MAE:        {test_metrics['avg_mae']:.4f}")
print(f"Test R² Score:   {test_metrics['avg_r2']:.4f}  ({test_metrics['avg_r2']*100:.2f}%)")
print("====================================================================")



                   CANONICAL EVALUATION RESULTS                     
Validation Loss: 0.4372
Test Loss:       6.2888
Test RMSE:       15527.4489
Test MAE:        6787.9867
Test R² Score:   0.7850  (78.50%)


In [6]:
# Gather per-month predictions and true values for visualization
test_preds_list = []
test_targets_list = []
test_periods_list = []

with torch.no_grad():
    H_test = H_val.detach()
    for snap in test_snaps:
        pred, H_test = model(snap, H_test)
        pred_raw = (pred * dataset.target_std.to(device) + dataset.target_mean.to(device)).cpu().numpy()
        target_raw = (snap.y * dataset.target_std.to(device) + dataset.target_mean.to(device)).cpu().numpy()
        mask = snap.active_mask.cpu().numpy()
        
        test_preds_list.append(pred_raw[mask])
        test_targets_list.append(target_raw[mask])
        test_periods_list.extend([snap.period] * int(mask.sum()))

all_preds_eval = np.concatenate(test_preds_list, axis=0)
all_targets_eval = np.concatenate(test_targets_list, axis=0)

df_test_eval = pd.DataFrame({
    "Period": test_periods_list,
    "Actual_Units": all_targets_eval[:, 0],
    "Pred_Units": all_preds_eval[:, 0],
    "Actual_Load": all_targets_eval[:, 1],
    "Pred_Load": all_preds_eval[:, 1]
})

df_test_eval["Residual_Units"] = df_test_eval["Actual_Units"] - df_test_eval["Pred_Units"]
df_test_eval["Residual_Load"] = df_test_eval["Actual_Load"] - df_test_eval["Pred_Load"]
df_test_eval.head(10)


In [7]:
sns.set_theme(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Units Demand Scatter
ax1 = axes[0, 0]
sns.scatterplot(data=df_test_eval, x="Actual_Units", y="Pred_Units", hue="Period", alpha=0.7, s=50, ax=ax1)
max_val_u = max(df_test_eval["Actual_Units"].max(), df_test_eval["Pred_Units"].max())
ax1.plot([0, max_val_u], [0, max_val_u], "r--", lw=2, label="Ideal (y=x)")
ax1.set_title("Units Demand (kWh) - Prediction vs Actual", fontweight="bold")
ax1.set_xlabel("Actual Monthly Units (kWh)")
ax1.set_ylabel("Predicted Monthly Units (kWh)")
ax1.legend(loc="upper left")

# 2. Peak Load Scatter
ax2 = axes[0, 1]
sns.scatterplot(data=df_test_eval, x="Actual_Load", y="Pred_Load", hue="Period", alpha=0.7, s=50, ax=ax2)
max_val_l = max(df_test_eval["Actual_Load"].max(), df_test_eval["Pred_Load"].max())
ax2.plot([0, max_val_l], [0, max_val_l], "r--", lw=2, label="Ideal (y=x)")
ax2.set_title("Peak Load (kW) - Prediction vs Actual", fontweight="bold")
ax2.set_xlabel("Actual Peak Load (kW)")
ax2.set_ylabel("Predicted Peak Load (kW)")
ax2.legend(loc="upper left")

# 3. Monthly Aggregate Demand Trend
ax3 = axes[1, 0]
monthly_summary = df_test_eval.groupby("Period")[["Actual_Units", "Pred_Units"]].sum().reset_index()
x_coords = np.arange(len(monthly_summary))
width = 0.35
ax3.bar(x_coords - width/2, monthly_summary["Actual_Units"], width, label="Actual Total Units", color="#2b5c8f")
ax3.bar(x_coords + width/2, monthly_summary["Pred_Units"], width, label="Predicted Total Units", color="#d95f02")
ax3.set_xticks(x_coords)
ax3.set_xticklabels(monthly_summary["Period"])
ax3.set_title("Total Monthly Statewide Demand Forecast (Aug-Dec 2025)", fontweight="bold")
ax3.set_ylabel("Aggregated Units (kWh)")
ax3.legend()

# 4. Residual Distribution
ax4 = axes[1, 1]
sns.histplot(df_test_eval["Residual_Units"], kde=True, color="#7570b3", ax=ax4)
ax4.axvline(0, color="red", linestyle="--", lw=2)
ax4.set_title("Units Prediction Residual Error Distribution", fontweight="bold")
ax4.set_xlabel("Residual (Actual - Predicted kWh)")

plt.tight_layout()
plt.show()


### Canonical Demonstration Summary
- **Architecture**: Stanford ROLAND GNN (`ResidualEdgeConv` + `GRUUpdater`)
- **Graph Nodes**: 934 EVCS Charging Stations across 33 Districts of Telangana
- **Temporal Sequence**: 36 Monthly Graph Snapshots (Jan 2023 - Dec 2025)
- **Pre-trained Checkpoint**: `final_model/roland_final_submission_80_02.pt` / `roland_final_weights_80_02.pt`
- **Evaluation Status**: Inference performed cleanly with ZERO shape mismatches.
